# WO-08 — WO-07 decisions, six new units, and the coastline
Cell 2 writes the ten filled_profiles. Cells 3 to 5 are the cutoff evidence for S1.3 (dropped; distance limits 25, 50, 100 km) and S1.4 (share 5, 10, 20%; 10% chosen). Cell 6 checks the snap (S8.3). Cells 7 to 9 are the coastline evidence and check (S7.2, S7.3; S7.1a dropped). Wording is `profile_templates.md`; rules are in `scripts/edop/whg/` (`wo08_units.py` for the new units). Results: `docs/design/dashboard/wo08_results.md`. Read-only except `wo08_filled_profiles.md` and the CSVs under `output/edop/whg/`.

In [2]:
%matplotlib inline
# Cell 1 — Setup: wording file, server check, saved ring, flow and river tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV
from scripts.edop.whg import wo08_units as U8

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit c7be9db | server: {'status': 'ok'} | wording: profile_templates.md v15 (last commit c7be9db; edited since: no); script wo04_fill.py (last commit c7be9db; edited since: no); script wo06_rings.py (last commit e516acb; edited since: no); script wo07_rivers.py (last commit 1a98688; edited since: yes); script wo07_classes.py (last commit 7090f1c; edited since: no); script wo08_units.py (last commit c7be9db; edited since: no) | ring table: 315 basins | river table: 574 | test places: 10 | sample points: 306


In [3]:
# Cell 2 — The ten test places: the filled_profiles (written to docs/design/dashboard/wo08_filled_profiles.md) and the sentences of the new units
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo08_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-08: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo08_new_units.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Units with nothing true to say are silent. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
L = []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    Un = W.fill_units(F)
    TEST[name] = (F, Un)
    parts.append(W.render_filled_profile(name, F, Un)); parts.append('---'); parts.append('')
    L.append(name)
    for u in ('S1.4', 'S2.1b', 'S2.3', 'S3.4', 'S5.2'):
        if u in Un:
            L.append(f'   {u}: ' + (Un[u]['text'] if Un[u]['ok'] else f'[silent: {Un[u]["why"]}]'))
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + '\n'.join(L))

wrote wo08_filled_profiles.md | profile_templates.md v15 (last commit c7be9db; edited since: no); script wo04_fill.py (last commit c7be9db; edited since: no); script wo06_rings.py (last commit e516acb; edited since: no); script wo07_rivers.py (last commit 1a98688; edited since: yes); script wo07_classes.py (last commit 7090f1c; edited since: no); script wo08_units.py (last commit c7be9db; edited since: no) | 21s

Timbuktu
   S1.4: [silent: no other ecoregion covers 10% of the basin or more]
   S2.1b: strongly concentrated, with August the wettest month
   S2.3: Compared with the five basins around it, its basin receives less precipitation than three and is cooler than three.
   S3.4: Lakes cover about 0.4% of the basin and wetlands about 37%.
   S5.2: Its basin has less pasture than three of the five basins around it.
Tbilisi
   S1.4: Its basin also takes in parts of the Eastern Anatolian montane steppe ecoregion.
   S2.1b: moderately seasonal, with May the wettest month
   S2.3: Compa

In [2]:
# Cell 3 — B3: distance to the next ecoregion for the ten places and the 306 sample points; saved to a CSV (resumable; about 1 to 2 minutes)
OUT3 = C.OUT / 'wo08_next_eco.csv'
done = pd.read_csv(OUT3, dtype={'key': str}) if OUT3.exists() else pd.DataFrame(columns=['key'])
have = set(done['key'])
pts = ([(n, 'test', la, lo) for n, (la, lo) in C.PLACES.items()]
       + [(str(int(s['hybas_id_l6'])), 'sample', float(s['lat']), float(s['lon'])) for s in SAMPLE])
inv = U8.invalid_ids(conn)
rows, t0 = [], time.perf_counter()
for k, (key, grp, la, lo) in enumerate(pts, 1):
    if key in have:
        continue
    pe = conn.execute('SELECT e.eco_id::bigint FROM gaz."Ecoregions2017" e WHERE e.geom && ST_SetSRID(ST_MakePoint(%s, %s), 4326) AND ST_Intersects(CASE WHEN e.eco_id = ANY(%s) THEN ST_MakeValid(e.geom) ELSE e.geom END, ST_SetSRID(ST_MakePoint(%s, %s), 4326)) LIMIT 1', (lo, la, inv, lo, la)).fetchone()
    b = conn.execute('SELECT hybas_id::bigint FROM public.basin06 WHERE ST_Intersects(geom, ST_SetSRID(ST_MakePoint(%s, %s), 4326)) LIMIT 1', (lo, la)).fetchone()
    top = U8.eco_shares(conn, b[0])[:1] if b else []
    own = int(pe[0]) if pe else (top[0][0] if top else None)
    nx = U8.next_ecoregion(conn, la, lo, own) if own is not None else None
    rows.append(dict(key=key, group=grp, lat=la, lon=lo, hid=b[0] if b else None, own=own, own_from='point' if pe else ('basin' if top else ''),
                     basin_top=top[0][0] if top else None, next_id=nx[0] if nx else None, next_name=nx[1] if nx else None, km=nx[2] if nx else None))
    if len(rows) % 50 == 0:
        print(f'  {len(rows)} done in {time.perf_counter() - t0:.0f}s')
if rows:
    done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
    done.to_csv(OUT3, index=False)
print(f'rows saved: {len(done)} | this run {len(rows)} in {time.perf_counter() - t0:.0f}s')

rows saved: 316 | this run 316 in 307s


In [3]:
# Cell 4 — B3 report: how often a next ecoregion lies within D km, and how often it is the ecoregion S1.2c already names
N3 = pd.read_csv(C.OUT / 'wo08_next_eco.csv', dtype={'key': str})
S3 = N3[(N3['group'] == 'sample') & N3['hid'].notna()]
n = len(S3)
L = [f'sample points with a basin: {n} | with an ecoregion to measure from: {int(S3["own"].notna().sum())} | point in no ecoregion polygon (basin majority used): {int((S3["own_from"] == "basin").sum())}', '']
L.append('distance from the point to the nearest other ecoregion, km: ' + str(S3['km'].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_dict()))
L.append('')
for D in (25, 50, 100):
    k = int((S3['km'] <= D).sum())
    L.append(f'within {D:>3} km: the sentence would speak at {k} of {n} ({100 * k / n:.0f}%); silent at {n - k}')
L.append(f'within 1 km ("less than a kilometre"): {int((S3["km"] < 1).sum())}')
same = S3[S3['next_id'] == S3['basin_top']]
L.append('')
L.append(f'the nearest other ecoregion is also the basin\'s largest-share ecoregion (so S1.2c, where it speaks, names it too): {len(same)} of {n}')
L.append('')
L.append('the ten places:')
T3 = N3[N3['group'] == 'test'][['key', 'own', 'own_from', 'next_name', 'km']].copy()
T3['km'] = T3['km'].round(1)
L.append(T3.to_string(index=False))
print('\n'.join(L))

sample points with a basin: 306 | with an ecoregion to measure from: 304 | point in no ecoregion polygon (basin majority used): 1

distance from the point to the nearest other ecoregion, km: {'count': 284.0, 'mean': 58.7, 'std': 67.3, 'min': 0.2, '10%': 3.4, '25%': 9.7, '50%': 34.2, '75%': 85.8, '90%': 143.8, 'max': 395.9}

within  25 km: the sentence would speak at 123 of 306 (40%); silent at 183
within  50 km: the sentence would speak at 167 of 306 (55%); silent at 139
within 100 km: the sentence would speak at 226 of 306 (74%); silent at 80
within 1 km ("less than a kilometre"): 5

the nearest other ecoregion is also the basin's largest-share ecoregion (so S1.2c, where it speaks, names it too): 13 of 306

the ten places:
             key   own own_from                                 next_name    km
        Timbuktu  53.0    point         Inner Niger Delta flooded savanna   4.3
         Tbilisi 812.0    point                    Caucasus mixed forests   2.8
        Santa Fe 429.0    

In [4]:
# Cell 5 — B4: the basin's other ecoregions at 5, 10 and 20% (shares from WO-03's q3_basin_eco.csv; the point's ecoregion from the table of Cell 3)
EB = pd.read_csv(C.OUT / 'q3_basin_eco.csv')
EB = EB[EB['eco_id'] >= 0]
NAME = dict(conn.execute('SELECT eco_id::bigint, eco_name FROM gaz."Ecoregions2017"').fetchall())
EB['share'] = EB['km2'] / EB['basin_km2']
N3 = pd.read_csv(C.OUT / 'wo08_next_eco.csv', dtype={'key': str})
rows = []
for r in N3[N3['hid'].notna()].itertuples():
    sub = EB[(EB['basin_id'] == int(r.hid)) & (EB['share'] > 0)].sort_values('share', ascending=False)
    if sub.empty:
        rows.append(dict(key=r.key, group=r.group, have_shares=False)); continue
    top = int(sub['eco_id'].iloc[0])
    skip = {top, int(r.own) if pd.notna(r.own) else None}
    oth = sub[~sub['eco_id'].isin(skip)]
    d = dict(key=r.key, group=r.group, have_shares=True)
    for t in (5, 10, 20):
        d[f'n{t}'] = int((oth['share'] >= t / 100).sum())
    d['names10'] = ', '.join(f"{NAME[int(e)]} {s:.0%}" for e, s in zip(oth['eco_id'], oth['share']) if s >= .10)
    d['names5'] = ', '.join(f"{NAME[int(e)]} {s:.0%}" for e, s in zip(oth['eco_id'], oth['share']) if s >= .05)
    rows.append(d)
B4 = pd.DataFrame(rows)
B4.to_csv(C.OUT / 'wo08_other_eco.csv', index=False)
S4 = B4[(B4['group'] == 'sample') & B4['have_shares']]
n = len(S4)
L = [f'sample basins with ecoregion shares: {n} of {int((B4["group"] == "sample").sum())}', '']
for t in (5, 10, 20):
    vc = S4[f'n{t}'].clip(upper=3).value_counts().sort_index()
    L.append(f'at {t:>2}%: other ecoregions per basin (0, 1, 2, 3 or more): ' + str({int(k): int(v) for k, v in vc.items()}) + f' | the sentence would speak at {int((S4[f"n{t}"] > 0).sum())} of {n} ({100 * (S4[f"n{t}"] > 0).sum() / n:.0f}%)')
L.append('')
L.append('the ten places (count at 5% / 10% / 20%, then names at 10%):')
for r in B4[B4['group'] == 'test'].itertuples():
    L.append(f"   {r.key}: {getattr(r, 'n5', '-')} / {getattr(r, 'n10', '-')} / {getattr(r, 'n20', '-')} | {getattr(r, 'names10', '')}")
print('\n'.join(L))

sample basins with ecoregion shares: 298 of 306

at  5%: other ecoregions per basin (0, 1, 2, 3 or more): {0: 153, 1: 113, 2: 27, 3: 5} | the sentence would speak at 145 of 298 (49%)
at 10%: other ecoregions per basin (0, 1, 2, 3 or more): {0: 177, 1: 99, 2: 20, 3: 2} | the sentence would speak at 121 of 298 (41%)
at 20%: other ecoregions per basin (0, 1, 2, 3 or more): {0: 211, 1: 81, 2: 6} | the sentence would speak at 87 of 298 (29%)

the ten places (count at 5% / 10% / 20%, then names at 10%):
   Timbuktu: 0.0 / 0.0 / 0.0 | 
   Tbilisi: 1.0 / 1.0 / 0.0 | Eastern Anatolian montane steppe 16%
   Santa Fe: 3.0 / 2.0 / 1.0 | Colorado Rockies forests 25%, Arizona Mountains forests 11%
   Kaifeng: 0.0 / 0.0 / 0.0 | 
   Lviv: 0.0 / 0.0 / 0.0 | 
   Poltava: 0.0 / 0.0 / 0.0 | 
   Chaco Canyon: 1.0 / 1.0 / 0.0 | Arizona Mountains forests 10%
   Alexandria: 1.0 / 1.0 / 0.0 | Nile Delta flooded savanna 11%
   Utqiagvik: 0.0 / 0.0 / 0.0 | 


In [2]:
# Cell 6 — B5 edge cases: Akureyri and Levuka (just outside a basin, should be moved), Tristan da Cunha (no basin, should stay none), Timbuktu (inside, no move)
lv = conn.execute("SELECT ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities WHERE city ILIKE 'Levuka%'").fetchone()
EDGE = {'Akureyri': (65.68353, -18.0878), 'Levuka': (float(lv[0]), float(lv[1])), 'Tristan da Cunha': C.PLACES['Tristan da Cunha'], 'Timbuktu': C.PLACES['Timbuktu']}
L = []
for name, (la, lo) in EDGE.items():
    F = W.build_facts(la, lo, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name} ({la:.5f}, {lo:.5f}): no basin (nobasin text, no snap)'); continue
    Un = W.fill_units(F)
    s83 = Un['S8.3']
    L.append(f'{name} ({la:.5f}, {lo:.5f}): basin {F["hid"]} | moved {F["snap_m"]} m | S8.3: ' + (s83['text'] if s83['ok'] else f'[silent: {s83["why"]}]'))
    L.append(f'   S1.1b elevation {Un["S1.1b"]["text"] if Un["S1.1b"]["ok"] else "[" + Un["S1.1b"]["why"] + "]"} | pt ecoregion {F["pt_eco_name"]} | S8.1a {Un["S8.1a"]["text"]} km²')
print('\n'.join(L) + '\n\nwording: ' + W.templates_stamp())

Akureyri (65.68353, -18.08780): basin 2060058000 | moved 195.26091504 m | S8.3: The point lies just outside the basin data and was moved 195 m to the nearest basin.
   S1.1b elevation 7 | pt ecoregion None | S8.1a 6,554 km²
Levuka (-17.68401, 178.84013): basin 5060081550 | moved 4.11619567 m | S8.3: The point lies just outside the basin data and was moved 4 m to the nearest basin.
   S1.1b elevation 31 | pt ecoregion None | S8.1a 175,335 km²
Tristan da Cunha (-37.06800, -12.31200): no basin (nobasin text, no snap)
Timbuktu (16.77000, -3.01000): basin 1060551560 | moved None m | S8.3: [silent: the point is inside a basin]
   S1.1b elevation 272 | pt ecoregion Sahelian Acacia savanna | S8.1a 3,826 km²

wording: profile_templates.md v13 (last commit 7efab25; edited since: yes); script wo04_fill.py (last commit 7efab25; edited since: yes); script wo06_rings.py (last commit e516acb; edited since: no); script wo07_rivers.py (last commit 1a98688; edited since: no); script wo07_classes.py (las

In [2]:
# Cell 7 — B6: distances to the coastline for the ten places, the 306 sample points and the 258 WH Cities (about 1 to 2 minutes); saved to a CSV. Run the Caspian both ways
Q6 = pd.read_csv(C.OUT / 'q6_points.csv', dtype={'id': str})
Q6 = Q6[Q6['hybas'].notna()].copy()
CASP_TERMINAL = {int(r[0]) for r in conn.execute("""SELECT t.hybas_id::bigint FROM public.basin06 t WHERE t.hybas_id = t.main_bas
    AND ST_DWithin(t.geom::geography, (SELECT geom::geography FROM gaz.ne_coastline_10m WHERE gid = 958), 2000)""").fetchall()}
rows, t0 = [], time.perf_counter()
for k, r in enumerate(Q6.itertuples(), 1):
    hid = int(r.hybas)
    b = conn.execute('SELECT coast, endo, main_bas::bigint FROM public.basin06 WHERE hybas_id = %s', (hid,)).fetchone()
    d = dict(set=r.set, id=r.id, label=r.label, lat=r.lat, lon=r.lon, hid=hid, coast_flag=b[0], endo=b[1], main_bas=b[2],
             into_caspian=b[2] in CASP_TERMINAL, d_boundary_km=r.d_boundary_km, nn_km=r.nn_km)
    for tag, casp in (('all', True), ('nocasp', False)):
        d[f'd_{tag}'] = U8.coast_km(conn, r.lat, r.lon, caspian=casp)
        d[f'o_{tag}'], d[f's_{tag}'] = U8.basin_coast(conn, hid, r.lat, r.lon, caspian=casp)
    rows.append(d)
    if k % 100 == 0:
        print(f'  {k} done in {time.perf_counter() - t0:.0f}s')
CO = pd.DataFrame(rows)
CO.to_csv(C.OUT / 'wo08_coast.csv', index=False)
print(f'rows saved: {len(CO)} | {CO.groupby("set").size().to_dict()} | basins draining into the Caspian among them: {int(CO["into_caspian"].sum())} | {time.perf_counter() - t0:.0f}s')

rows saved: 569 | {'sample': 306, 'test': 9, 'wh_cities': 254} | basins draining into the Caspian among them: 16 | 59s


In [ ]:
# Cell 8 — B6 report: S7.1a distances, S7.2 and S7.3 against the signature's coast flag and WO-03's rule, and what the Caspian changes
CO = pd.read_csv(C.OUT / 'wo08_coast.csv', dtype={'id': str})
CO['q6_coast'] = CO['nn_km'].isna() | (CO['d_boundary_km'] < CO['nn_km'] - 1e-6)
big = CO[CO['set'].isin(['sample', 'wh_cities'])]
L = []
L.append('S7.1a: distance from the point to the nearest coastline, km (Caspian counted as coast)')
for s in ('sample', 'wh_cities'):
    x = CO[CO['set'] == s]
    L.append(f'  {s} ({len(x)}): ' + str(x['d_all'].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_dict()) + f' | under 1 km: {int((x["d_all"] < 1).sum())} | under 10 km: {int((x["d_all"] < 10).sum())} | none within 5 degrees: {int(x["d_all"].isna().sum())}')
L.append('')
L.append("S7.2: the signature's coast flag against 'the basin's outline lies within T km of the coastline' (sample and WH Cities, Caspian counted)")
for T in (1, 2, 5):
    ct = pd.crosstab(big['coast_flag'], big['o_all'].le(T).map({True: 'layer: touches', False: 'layer: does not'}))
    L.append(f'  T = {T} km:\n' + ct.to_string())
L.append('')
L.append("S7.3: WO-03's rule (nearest outline closer than any neighbouring basin) against 'the nearest outline spot lies within T km of the coastline'")
for T in (1, 2, 5):
    ct = pd.crosstab(big['q6_coast'].map({True: 'WO-03: coast', False: 'WO-03: divide'}), big['s_all'].le(T).map({True: 'layer: coast', False: 'layer: not'}))
    L.append(f'  T = {T} km:\n' + ct.to_string())
L.append('')
ten = CO[CO['set'] == 'test'][['label', 'coast_flag', 'd_all', 'o_all', 's_all', 'q6_coast', 'into_caspian']].copy()
for c in ('d_all', 'o_all', 's_all'):
    ten[c] = ten[c].round(1)
L.append('the ten places:\n' + ten.to_string(index=False))
L.append('')
cas = big[big['into_caspian']]
L.append(f'THE CASPIAN. Basins that drain into it: {len(cas)} of {len(big)} (sample {int((cas["set"] == "sample").sum())}, WH Cities {int((cas["set"] == "wh_cities").sum())}); the ten places: ' + (', '.join(CO[(CO["set"] == "test") & CO["into_caspian"]]['label']) or 'none'))
L.append(f'  of those, the signature says endo=0 (drains to the sea): {int((cas["endo"] == 0).sum())}; endo=1 or 2: {int((cas["endo"] != 0).sum())}; coast flag 1: {int((cas["coast_flag"] == 1).sum())}')
d_chg = big[big['d_all'].fillna(-1).round(1) != big['d_nocasp'].fillna(-1).round(1)]
d_chg = d_chg[d_chg['d_all'].notna()]
L.append(f'  points whose distance to the coast changes if the Caspian is not coast (a distance within 5 degrees that becomes none or larger): {len(d_chg)} (' + ', '.join(f'{r.label} {r.d_all:.0f} to ' + ('none' if pd.isna(r.d_nocasp) else f'{r.d_nocasp:.0f}') + ' km' for r in d_chg.head(12).itertuples()) + ')')
for T in (2,):
    t_chg = big[(big['o_all'].le(T)) != (big['o_nocasp'].le(T))]
    L.append(f'  basins whose outline touches the coast at {T} km only through the Caspian: {len(t_chg)}')
print('\n'.join(L))

In [2]:
# Cell 9 — B6 check: the Coast and reach sentences (S7.2, S7.3) at the ten places and at six coastal cities (the Caspian, ports, an estuary)
extra = {}
for like in ('Baku%', 'Aktau%', 'Porto', 'Rotterdam%', 'Stockholm%', 'Hamburg%'):
    r = conn.execute("SELECT city || ', ' || country, ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities WHERE city ILIKE %s LIMIT 1", (like,)).fetchone()
    if r:
        extra[r[0]] = (float(r[1]), float(r[2]))
L = []
for name, (la, lo) in {**C.PLACES, **extra}.items():
    F = W.build_facts(la, lo, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    Un = W.fill_units(F)
    L.append(f'{name} (coast flag {W.val(F, "coast_flag")}; outline {F["coast"]["o"]} km, nearest edge {F["coast"]["s"]} km from the coastline)')
    for u in ('S7.2', 'S7.3'):
        L.append(f'   {u}: ' + (Un[u]['text'] if Un[u]['ok'] else f'[silent: {Un[u]["why"]}]'))
print('\n'.join(L) + '\n\nwording: ' + W.templates_stamp())

Timbuktu (coast flag 0; outline None km, nearest edge None km from the coastline)
   S7.2: [silent: basin does not touch the sea (optional part not stated)]
   S7.3: [silent: the nearest edge of the basin is not the seacoast (optional part not stated)]
Tbilisi (coast flag 0; outline 45.42099207396 km, nearest edge 246.96685547191998 km from the coastline)
   S7.2: [silent: basin does not touch the sea (optional part not stated)]
   S7.3: [silent: the nearest edge of the basin is not the seacoast (optional part not stated)]
Santa Fe (coast flag 0; outline None km, nearest edge None km from the coastline)
   S7.2: [silent: basin does not touch the sea (optional part not stated)]
   S7.3: [silent: the nearest edge of the basin is not the seacoast (optional part not stated)]
Kaifeng (coast flag 0; outline 232.94299101822 km, nearest edge 446.10642938182997 km from the coastline)
   S7.2: [silent: basin does not touch the sea (optional part not stated)]
   S7.3: [silent: the nearest edge of